# Pipeline RAG completo — Regulamento Geral da Graduação da UFRPE

Este é o notebook principal. O corpus é o **Regulamento Geral da Graduação (PREG/UFRPE, 2024)**, dividido em 30 documentos temáticos por `split_regulamento.py` (um ou mais Títulos/Capítulos por documento, por exemplo matrícula, suspensão de programa, monitoria e glossário). O notebook lê os PDFs de `documents/`, cria chunks de tamanho fixo com overlap, persiste embeddings no ChromaDB e compara os dois métodos de retrieval:

- **esparso:** TF-IDF com similaridade cosseno;
- **denso:** embeddings Gemini com similaridade cosseno no ChromaDB.

Ao final, três perguntas são respondidas sem retrieval e com os dois retrievers. Uma pergunta demonstrativa adicional completa as 10 respostas da avaliação manual.

Para regenerar o corpus: `python split_regulamento.py "Regulamento geral PREG 2024.pdf" documents`.

## 1. Modelos, caminhos e banco

In [ ]:
import os
from pathlib import Path
from pprint import pprint

import chromadb
from dotenv import load_dotenv
from google import genai

import rag_utils

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "documents"
CHROMA_DIR = PROJECT_ROOT / "db" / "chroma"
COLLECTION_NAME = "regulamento_ufrpe_v1"
EMBED_MODEL = "gemini-embedding-001"
GEN_MODEL = "gemini-3.1-flash-lite"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 5

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ.get("GEMINI_API_KEY")
assert API_KEY, "Defina GEMINI_API_KEY no arquivo .env da raiz do projeto."

client = genai.Client(api_key=API_KEY)
embedder = rag_utils.GeminiEmbedder(client, model=EMBED_MODEL)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
)
print(f"Documentos: {DATA_DIR}")
print(f"Banco: {CHROMA_DIR}")

Documentos: C:\Users\User\Desktop\Mineração\rag-practice\documents
Banco: C:\Users\User\Desktop\Mineração\rag-practice\db\chroma


## 2. Ingestão e indexação

A indexação é retomável: chunks já presentes no banco não são enviados novamente à API de embeddings.

In [ ]:
pdf_paths = rag_utils.iter_pdfs(DATA_DIR)
records = rag_utils.load_corpus(DATA_DIR, CHUNK_SIZE, CHUNK_OVERLAP)
assert records, f"Nenhum conteúdo PDF foi encontrado em {DATA_DIR}."

index_stats = rag_utils.index_dense_collection(collection, records, embedder)
print(f"PDFs: {len(pdf_paths)}")
print(f"Chunks: {len(records)}")
print(index_stats)

PDFs: 30
Chunks: 274
{'total': 274, 'already_indexed': 0, 'indexed_now': 274}


## 3. Os dois métodos de retrieval

Ambos usam exatamente os mesmos chunks. O método denso consulta os vetores persistidos no Chroma; o TF-IDF monta um índice esparso local a partir dos textos guardados no mesmo banco.

In [ ]:
indexed_records = rag_utils.records_from_collection(collection)
dense_retriever = rag_utils.DenseRetriever(collection, embedder)
tfidf_retriever = rag_utils.TfidfRetriever(indexed_records)

dense_rag = rag_utils.RAGAssistant(client, dense_retriever, GEN_MODEL)
tfidf_rag = rag_utils.RAGAssistant(client, tfidf_retriever, GEN_MODEL)

In [ ]:
sample_question = "Por quanto tempo um(a) discente pode manter a condição de aluno especial?"

print("Ranking denso:")
pprint(rag_utils.hits_as_rows(dense_retriever.retrieve(sample_question, TOP_K)), sort_dicts=False)
print("\nRanking TF-IDF:")
pprint(rag_utils.hits_as_rows(tfidf_retriever.retrieve(sample_question, TOP_K)), sort_dicts=False)

Ranking denso:


[{'rank': 1,
  'cosine_similarity': 0.7459,
  'paper': '10_aluno_especial_ordinario',
  'page': 1,
  'chunk': 0,
  'snippet': 'UFRPE — Regulamento Geral da Graduação (PREG, 2024)\n'
             'Alunos especiais de graduação e aluno especial ordinário\n'
             'TÍTULO IX\n'
             'DOS ALUNOS ESPECIAIS DE GRADUAÇÃO\n'
             'Art. 180. O(A) discente de graduação admitido(a) através de '
             'qualque…'},
 {'rank': 2,
  'cosine_similarity': 0.7409,
  'paper': '10_aluno_especial_ordinario',
  'page': 2,
  'chunk': 2,
  'snippet': 'is ordinários, além das restrições que se aplicam a todos(as) '
             'os(as)\n'
             'alunos(as) especiais, definidas no Art. 181, não podem: - '
             'solicitar e outros bens da UFRPE;\n'
             'I - realizar estágio;\n'
             'II - matricular-se em componentes cur…'},
 {'rank': 3,
  'cosine_similarity': 0.7323,
  'paper': '12_complementacao_movimentacao_temporaria',
  'page': 1,
  'chunk': 3,
  

## 4. Geração fundamentada

O prompt contém os top-k chunks completos e exige citações no formato `[documento, p. N, trecho M]`.

In [ ]:
demo_result = dense_rag.ask(sample_question, k=TOP_K)
print(demo_result.answer)

A condição de aluno especial tem duração limitada a quatro períodos letivos consecutivos [10_aluno_especial_ordinario, p. 1, trecho 0].


## 5. Três perguntas: sem retrieval, RAG denso e RAG TF-IDF

Esta célula gera nove respostas comparáveis. Somadas à demonstração acima, são as 10 respostas usadas na avaliação manual.

In [ ]:
QUESTIONS = [
    "Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?",
    "Em que situação um(a) discente é colocado(a) no regime de observação do desempenho acadêmico?",
    "Como é calculada a nota final da seleção para monitoria?",
]

generated_answers = [
    {"question": sample_question, "method": "dense", "answer": demo_result.answer}
]

for question in QUESTIONS:
    baseline = rag_utils.answer_without_retrieval(client, question, GEN_MODEL)
    dense_result = dense_rag.ask(question, k=TOP_K)
    tfidf_result = tfidf_rag.ask(question, k=TOP_K)

    generated_answers.extend([
        {"question": question, "method": "no_retrieval", "answer": baseline},
        {"question": question, "method": "dense", "answer": dense_result.answer},
        {"question": question, "method": "tfidf", "answer": tfidf_result.answer},
    ])

    print(f"\n{'=' * 90}\nPERGUNTA: {question}")
    print(f"\nSEM RETRIEVAL\n{baseline}")
    print("\nCHUNKS — DENSO")
    pprint(rag_utils.hits_as_rows(dense_result.hits), sort_dicts=False)
    print(f"\nRAG DENSO\n{dense_result.answer}")
    print("\nCHUNKS — TF-IDF")
    pprint(rag_utils.hits_as_rows(tfidf_result.hits), sort_dicts=False)
    print(f"\nRAG TF-IDF\n{tfidf_result.answer}")

assert len(generated_answers) == 10


PERGUNTA: Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?

SEM RETRIEVAL
Para responder à sua pergunta com precisão, é necessário esclarecer que as regras sobre suspensão de programas (seja acadêmico, de benefícios governamentais ou de planos de saúde) **variam drasticamente dependendo do regulamento específico** de cada instituição ou contrato.

Como você não especificou a qual programa se refere, aqui estão as diretrizes gerais aplicadas aos cenários mais comuns:

### 1. Programas Acadêmicos (Graduação/Pós-Graduação)
*   **Limite máximo:** Geralmente, as universidades permitem a suspensão (trancamento) por um período total que varia entre **2 a 4 semestres letivos** ao longo de todo o curso. Algumas instituições permitem que esse período seja contínuo ou intercalado.
*   **Prazo para solicitação:** Deve ser feita, obrigatoriamente, **antes do início do período letivo** ou dentro do prazo estipulado no calendário acadêmico (geralmente na


PERGUNTA: Em que situação um(a) discente é colocado(a) no regime de observação do desempenho acadêmico?

SEM RETRIEVAL
O regime de observação do desempenho acadêmico (frequentemente chamado de "alerta" ou "condicional") é uma medida administrativa aplicada por instituições de ensino superior quando o estudante apresenta um rendimento acadêmico abaixo dos padrões mínimos exigidos pela universidade.

Embora as regras específicas variem conforme o regimento interno de cada instituição, a situação ocorre, de modo geral, nos seguintes cenários:

1.  **Baixo Rendimento Acadêmico (IRA/Coeficiente):** Quando o Índice de Rendimento Acadêmico (IRA), Coeficiente de Rendimento (CR) ou média global do aluno cai abaixo de um valor mínimo estabelecido (por exemplo, abaixo de 5,0 ou 6,0, dependendo da escala da instituição).
2.  **Reprovação Excessiva:** Quando o estudante é reprovado em um número elevado de disciplinas em um único período letivo ou acumula um percentual de reprovações que compromete


PERGUNTA: Como é calculada a nota final da seleção para monitoria?

SEM RETRIEVAL
O cálculo da nota final em uma seleção de monitoria não possui um padrão único, pois **depende exclusivamente do edital específico** publicado pela instituição de ensino ou pelo departamento responsável.

No entanto, na maioria das universidades brasileiras, a nota final é composta pela soma ponderada de diferentes etapas. As mais comuns são:

### 1. Componentes comuns da nota
*   **Média Acadêmica (ou Coeficiente de Rendimento):** É a nota obtida nas disciplinas cursadas pelo aluno. Geralmente, tem um peso fixo na nota final para garantir que o monitor tenha bom desempenho na matéria que irá lecionar.
*   **Prova Escrita ou Teórica:** Avalia o conhecimento técnico do candidato sobre o conteúdo da disciplina.
*   **Prova Prática ou Didática:** Avalia a capacidade do aluno de transmitir o conhecimento (ex: uma aula expositiva de 10 a 15 minutos sobre um tema sorteado).
*   **Entrevista:** Avalia a disponi

## 6. Avaliação manual das 10 respostas

Preencha `True`/`False` depois de ler cada resposta e, nos casos com retrieval, os chunks exibidos. Para a baseline sem retrieval, suporte e citação não se aplicam.

In [ ]:
evaluation_sheet = [
    {
        "answer_id": index,
        "question": item["question"],
        "method": item["method"],
        "correct": None,
        "supported_by_chunks": None if item["method"] != "no_retrieval" else "N/A",
        "cites_right_chunk": None if item["method"] != "no_retrieval" else "N/A",
        "notes": "",
    }
    for index, item in enumerate(generated_answers, start=1)
]
evaluation_sheet

[{'answer_id': 1,
  'question': 'Por quanto tempo um(a) discente pode manter a condição de aluno especial?',
  'method': 'dense',
  'correct': None,
  'supported_by_chunks': None,
  'cites_right_chunk': None,
  'notes': ''},
 {'answer_id': 2,
  'question': 'Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?',
  'method': 'no_retrieval',
  'correct': None,
  'supported_by_chunks': 'N/A',
  'cites_right_chunk': 'N/A',
  'notes': ''},
 {'answer_id': 3,
  'question': 'Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?',
  'method': 'dense',
  'correct': None,
  'supported_by_chunks': None,
  'cites_right_chunk': None,
  'notes': ''},
 {'answer_id': 4,
  'question': 'Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?',
  'method': 'tfidf',
  'correct': None,
  'supported_by_chunks': None,
  'cites_right_chunk': None,
  'notes': ''},
 {'answer_id': 5,
  'questio

In [2]:
# Criar a planilha de avaliação sem executar o RAG

perguntas = [
    "Por quanto tempo um(a) discente pode manter a condição de aluno especial?",
    "Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?",
    "Em que situação um(a) discente é colocado(a) no regime de observação do desempenho acadêmico?",
    "Como é calculada a nota final da seleção para monitoria?"
]

casos = [
    (1, perguntas[0], "dense"),
    (2, perguntas[1], "no_retrieval"),
    (3, perguntas[1], "dense"),
    (4, perguntas[1], "tfidf"),
    (5, perguntas[2], "no_retrieval"),
    (6, perguntas[2], "dense"),
    (7, perguntas[2], "tfidf"),
    (8, perguntas[3], "no_retrieval"),
    (9, perguntas[3], "dense"),
    (10, perguntas[3], "tfidf"),
]

evaluation_sheet = [
    {
        "answer_id": num,
        "question": pergunta,
        "method": metodo,
        "correct": None,
        "supported_by_chunks": "N/A" if metodo == "no_retrieval" else None,
        "cites_right_chunk": "N/A" if metodo == "no_retrieval" else None,
        "notes": ""
    }
    for num, pergunta, metodo in casos
]

print("Planilha de avaliação criada com sucesso!")

Planilha de avaliação criada com sucesso!


In [5]:
# Avaliação 1 - Aluno especial (Dense)

evaluation_sheet[0].update({
    "correct": True,
    "supported_by_chunks": True,
    "cites_right_chunk": True,
    "notes": (
        "A resposta informa corretamente o limite de "
        "quatro períodos letivos consecutivos, conforme "
        "o Art. 180, §3º. O chunk recuperado sustenta "
        "a resposta e a citação corresponde à fonte."
    )
})

evaluation_sheet[0]

{'answer_id': 1,
 'question': 'Por quanto tempo um(a) discente pode manter a condição de aluno especial?',
 'method': 'dense',
 'correct': True,
 'supported_by_chunks': True,
 'cites_right_chunk': True,
 'notes': 'A resposta informa corretamente o limite de quatro períodos letivos consecutivos, conforme o Art. 180, §3º. O chunk recuperado sustenta a resposta e a citação corresponde à fonte.'}

In [3]:
# Avaliação 2 - Suspensão de programa (Sem Retrieval)

evaluation_sheet[1].update({
    "correct": False,
    "supported_by_chunks": "N/A",
    "cites_right_chunk": "N/A",
    "notes": (
        "A resposta é genérica e não apresenta corretamente "
        "as regras específicas da UFRPE. Conforme o Art. 249, "
        "o limite é de quatro períodos letivos regulares, "
        "consecutivos ou não, e a solicitação deve ocorrer "
        "dentro do prazo de 12 semanas após o início do "
        "período letivo. O modelo também incluiu informações "
        "irrelevantes sobre FIES e planos de saúde."
    )
})

evaluation_sheet[1]

{'answer_id': 2,
 'question': 'Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?',
 'method': 'no_retrieval',
 'correct': False,
 'supported_by_chunks': 'N/A',
 'cites_right_chunk': 'N/A',
 'notes': 'A resposta é genérica e não apresenta corretamente as regras específicas da UFRPE. Conforme o Art. 249, o limite é de quatro períodos letivos regulares, consecutivos ou não, e a solicitação deve ocorrer dentro do prazo de 12 semanas após o início do período letivo. O modelo também incluiu informações irrelevantes sobre FIES e planos de saúde.'}

In [8]:
# Avaliação 3 - Suspensão (Dense)
evaluation_sheet[2].update({
    "correct": True,
    "supported_by_chunks": True,
    "cites_right_chunk": True,
    "notes": (
        "Resposta correta conforme o Art. 249. "
        "O recuperador denso encontrou o trecho relevante "
        "no documento 15_trancamento_cancelamento_suspensao, "
        "página 1, trecho 2, citado corretamente."
    )
})
evaluation_sheet[2]

{'answer_id': 3,
 'question': 'Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?',
 'method': 'dense',
 'correct': True,
 'supported_by_chunks': True,
 'cites_right_chunk': True,
 'notes': 'Resposta correta conforme o Art. 249. O recuperador denso encontrou o trecho relevante no documento 15_trancamento_cancelamento_suspensao, página 1, trecho 2, citado corretamente.'}

In [11]:
# Avaliação 4 - Suspensão de programa (TF-IDF)

evaluation_sheet[3].update({
    "correct": True,
    "supported_by_chunks": True,
    "cites_right_chunk": True,
    "notes": (
        "Resposta correta conforme o Art. 249 do Regulamento "
        "Geral da Graduação da UFRPE. O TF-IDF recuperou "
        "o trecho relevante do documento "
        "15_trancamento_cancelamento_suspensao, "
        "página 1, trecho 2. A resposta está fundamentada "
        "e apresenta a citação correspondente."
    )
})
evaluation_sheet[3]
# Visualizar avaliações

{'answer_id': 4,
 'question': 'Qual é o limite máximo de suspensões de programa e até quando a suspensão deve ser solicitada?',
 'method': 'tfidf',
 'correct': True,
 'supported_by_chunks': True,
 'cites_right_chunk': True,
 'notes': 'Resposta correta conforme o Art. 249 do Regulamento Geral da Graduação da UFRPE. O TF-IDF recuperou o trecho relevante do documento 15_trancamento_cancelamento_suspensao, página 1, trecho 2. A resposta está fundamentada e apresenta a citação correspondente.'}

In [12]:
# Avaliação 5 - Regime de Observação (Sem Retrieval)

evaluation_sheet[4].update({
    "correct": False,
    "supported_by_chunks": "N/A",
    "cites_right_chunk": "N/A",
    "notes": (
        "A resposta é genérica e não apresenta o critério "
        "específico do Art. 322 do regulamento da UFRPE. "
        "O regime de observação ocorre após três ou mais "
        "insucessos, consecutivos ou não, em um mesmo "
        "componente curricular obrigatório ou equivalente. "
        "O modelo menciona fatores gerais, como baixo "
        "rendimento e abandono, sem identificar essa regra."
    )
})
evaluation_sheet[4]
# Visualizar avaliações atualizadas

{'answer_id': 5,
 'question': 'Em que situação um(a) discente é colocado(a) no regime de observação do desempenho acadêmico?',
 'method': 'no_retrieval',
 'correct': False,
 'supported_by_chunks': 'N/A',
 'cites_right_chunk': 'N/A',
 'notes': 'A resposta é genérica e não apresenta o critério específico do Art. 322 do regulamento da UFRPE. O regime de observação ocorre após três ou mais insucessos, consecutivos ou não, em um mesmo componente curricular obrigatório ou equivalente. O modelo menciona fatores gerais, como baixo rendimento e abandono, sem identificar essa regra.'}

In [14]:
# Avaliação 6 - Regime de Observação (RAG Denso)

evaluation_sheet[5].update({
    "correct": True,
    "supported_by_chunks": True,
    "cites_right_chunk": True,
    "notes": (
        "A resposta está correta conforme o Art. 322 do "
        "Regulamento Geral da Graduação da UFRPE. "
        "O recuperador denso encontrou o trecho relevante "
        "no documento 23_observacao_e_cancelamento_de_programa, "
        "página 1, trecho 1, além do glossário, página 2, "
        "trecho 2. Ambos sustentam a regra dos três "
        "insucessos, e as citações correspondem aos "
        "chunks recuperados."
    )
})
evaluation_sheet[5]
# Visualizar avaliações atualizadas

{'answer_id': 6,
 'question': 'Em que situação um(a) discente é colocado(a) no regime de observação do desempenho acadêmico?',
 'method': 'dense',
 'correct': True,
 'supported_by_chunks': True,
 'cites_right_chunk': True,
 'notes': 'A resposta está correta conforme o Art. 322 do Regulamento Geral da Graduação da UFRPE. O recuperador denso encontrou o trecho relevante no documento 23_observacao_e_cancelamento_de_programa, página 1, trecho 1, além do glossário, página 2, trecho 2. Ambos sustentam a regra dos três insucessos, e as citações correspondem aos chunks recuperados.'}

In [17]:
# Avaliação 7 - Regime de Observação (TF-IDF)

evaluation_sheet[6].update({
    "correct": True,
    "supported_by_chunks": True,
    "cites_right_chunk": True,
    "notes": (
        "Resposta correta conforme o Art. 322 do Regulamento "
        "Geral da Graduação da UFRPE. O TF-IDF recuperou "
        "o trecho relevante do documento "
        "23_observacao_e_cancelamento_de_programa, "
        "página 1, trecho 1, e o glossário, página 2, "
        "trecho 2. Ambos apresentam a regra dos três "
        "insucessos, e as citações correspondem "
        "aos chunks recuperados."
    )
})
evaluation_sheet[6]
# Visualizar avaliações atualizadas

{'answer_id': 7,
 'question': 'Em que situação um(a) discente é colocado(a) no regime de observação do desempenho acadêmico?',
 'method': 'tfidf',
 'correct': True,
 'supported_by_chunks': True,
 'cites_right_chunk': True,
 'notes': 'Resposta correta conforme o Art. 322 do Regulamento Geral da Graduação da UFRPE. O TF-IDF recuperou o trecho relevante do documento 23_observacao_e_cancelamento_de_programa, página 1, trecho 1, e o glossário, página 2, trecho 2. Ambos apresentam a regra dos três insucessos, e as citações correspondem aos chunks recuperados.'}

In [19]:
# Avaliação 8 - Seleção para monitoria (Sem Retrieval)

evaluation_sheet[7].update({
    "correct": False,
    "supported_by_chunks": "N/A",
    "cites_right_chunk": "N/A",
    "notes": (
        "A resposta não apresenta a fórmula específica do "
        "regulamento da UFRPE. O modelo fornece um exemplo "
        "genérico de média ponderada com pesos 40%, 30% e 30%, "
        "enquanto a nota final da seleção para monitoria "
        "corresponde à média aritmética das notas da banca "
        "examinadora, média de conclusão (MC) e nota do "
        "estudante no componente curricular (NE)."
    )
})
evaluation_sheet[7]
# Visualizar avaliações atualizadas

{'answer_id': 8,
 'question': 'Como é calculada a nota final da seleção para monitoria?',
 'method': 'no_retrieval',
 'correct': False,
 'supported_by_chunks': 'N/A',
 'cites_right_chunk': 'N/A',
 'notes': 'A resposta não apresenta a fórmula específica do regulamento da UFRPE. O modelo fornece um exemplo genérico de média ponderada com pesos 40%, 30% e 30%, enquanto a nota final da seleção para monitoria corresponde à média aritmética das notas da banca examinadora, média de conclusão (MC) e nota do estudante no componente curricular (NE).'}

In [21]:
# Avaliação 9 - Seleção para Monitoria (RAG Denso)

evaluation_sheet[8].update({
    "correct": True,
    "supported_by_chunks": True,
    "cites_right_chunk": True,
    "notes": (
        "A resposta apresenta corretamente a média aritmética "
        "das três notas: banca examinadora, média de conclusão "
        "(MC) e nota no componente curricular (NE). "
        "O recuperador denso encontrou o documento 26_monitoria, "
        "página 2, trecho 0, que contém os componentes do cálculo. "
        "As citações correspondem a chunks recuperados, embora "
        "a referência à página 1, trecho 3, precise de confirmação "
        "com o conteúdo integral do chunk."
    )
})

evaluation_sheet[8]
# Visualizar avaliações


{'answer_id': 9,
 'question': 'Como é calculada a nota final da seleção para monitoria?',
 'method': 'dense',
 'correct': True,
 'supported_by_chunks': True,
 'cites_right_chunk': True,
 'notes': 'A resposta apresenta corretamente a média aritmética das três notas: banca examinadora, média de conclusão (MC) e nota no componente curricular (NE). O recuperador denso encontrou o documento 26_monitoria, página 2, trecho 0, que contém os componentes do cálculo. As citações correspondem a chunks recuperados, embora a referência à página 1, trecho 3, precise de confirmação com o conteúdo integral do chunk.'}

In [23]:
# Avaliação 10 - Seleção para Monitoria (TF-IDF)

evaluation_sheet[9].update({
    "correct": False,
    "supported_by_chunks": True,
    "cites_right_chunk": True,
    "notes": (
        "A resposta identifica corretamente os três componentes "
        "da nota final: nota da banca examinadora, média de "
        "conclusão (MC) e nota no componente curricular (NE). "
        "Entretanto, não informa explicitamente que a nota "
        "final é a média aritmética simples desses valores, "
        "deixando incompleto o cálculo solicitado. "
        "O TF-IDF recuperou o documento 26_monitoria, "
        "página 2, trecho 0, e a resposta cita corretamente "
        "a fonte utilizada. Classificada como incompleta "
        "pelo critério de correção estrito."
    )
})
evaluation_sheet[9]
# Exibir a tabela completa

{'answer_id': 10,
 'question': 'Como é calculada a nota final da seleção para monitoria?',
 'method': 'tfidf',
 'correct': False,
 'supported_by_chunks': True,
 'cites_right_chunk': True,
 'notes': 'A resposta identifica corretamente os três componentes da nota final: nota da banca examinadora, média de conclusão (MC) e nota no componente curricular (NE). Entretanto, não informa explicitamente que a nota final é a média aritmética simples desses valores, deixando incompleto o cálculo solicitado. O TF-IDF recuperou o documento 26_monitoria, página 2, trecho 0, e a resposta cita corretamente a fonte utilizada. Classificada como incompleta pelo critério de correção estrito.'}

In [24]:
import pandas as pd

pd.DataFrame(evaluation_sheet)

,answer_id,question,method,correct,supported_by_chunks,cites_right_chunk,notes
0,1,Por quanto tempo um(a) discente pode manter a ...,dense,True,True,True,A resposta informa corretamente o limite de qu...
1,2,Qual é o limite máximo de suspensões de progra...,no_retrieval,False,N/A,N/A,A resposta é genérica e não apresenta corretam...
2,3,Qual é o limite máximo de suspensões de progra...,dense,True,True,True,Resposta correta conforme o Art. 249. O recupe...
3,4,Qual é o limite máximo de suspensões de progra...,tfidf,True,True,True,Resposta correta conforme o Art. 249 do Regula...
4,5,Em que situação um(a) discente é colocado(a) n...,no_retrieval,False,N/A,N/A,A resposta é genérica e não apresenta o critér...
5,6,Em que situação um(a) discente é colocado(a) n...,dense,True,True,True,A resposta está correta conforme o Art. 322 do...
6,7,Em que situação um(a) discente é colocado(a) n...,tfidf,True,True,True,Resposta correta conforme o Art. 322 do Regula...
7,8,Como é calculada a nota final da seleção para ...,no_retrieval,False,N/A,N/A,A resposta não apresenta a fórmula específica ...
8,9,Como é calculada a nota final da seleção para ...,dense,True,True,True,A resposta apresenta corretamente a média arit...
9,10,Como é calculada a nota final da seleção para ...,tfidf,False,True,True,A resposta identifica corretamente os três com...


## 7. Diagnóstico de duas respostas erradas

Depois da avaliação, escolha dois `answer_id` incorretos. Marque **retrieval** se os chunks necessários não chegaram ao modelo; marque **generation** se os chunks eram adequados, mas a resposta os interpretou mal, ignorou evidências ou citou o trecho errado.

In [25]:
# Seção 7 - Diagnóstico de duas respostas incorretas

failure_analysis = [
    {
        "answer_id": 10,
        "failure_source": "generation",
        "justification": (
            "O TF-IDF recuperou o documento correto sobre monitoria, "
            "incluindo o chunk da página 2, trecho 0. "
            "Entretanto, a resposta identificou os três componentes "
            "da nota final sem informar que o cálculo utiliza a "
            "média aritmética simples. Trata-se de uma falha de "
            "geração por omissão de informação relevante, "
            "e não de recuperação."
        )
    },
    {
        "answer_id": 2,
        "failure_source": "generation",
        "justification": (
            "O modelo respondeu sem retrieval e apresentou regras "
            "genéricas sobre suspensão de programas. "
            "Não identificou o limite de quatro períodos letivos "
            "nem o prazo de doze semanas previsto no Art. 249 "
            "do regulamento da UFRPE. A falha ocorreu na geração "
            "sem contexto documental. Não é uma falha de retrieval, "
            "pois nenhum recuperador foi utilizado."
        )
    }
]

import pandas as pd
pd.DataFrame(failure_analysis)

,answer_id,failure_source,justification
0,10,generation,O TF-IDF recuperou o documento correto sobre m...
1,2,generation,O modelo respondeu sem retrieval e apresentou ...


## 8. Risco de busca responsável

**Atualidade (freshness):** o índice é uma fotografia do Regulamento Geral da Graduação de 2024 no momento da ingestão. O próprio regulamento já traz dispositivos alterados por resoluções posteriores ("redação dada pela Resolução..."). Se o CEPE aprovar uma nova resolução mudando, por exemplo, o limite de suspensões de programa ou os critérios do RODA, o sistema continuará respondendo a regra antiga com citação e aparência de autoridade, e um(a) discente pode perder um prazo ou tomar uma decisão acadêmica errada.

**Mitigação:** registrar no metadado de cada chunk a versão e a data do regulamento, mostrar essa data junto de toda resposta ("segundo o Regulamento PREG 2024") e reindexar automaticamente quando a PREG publicar uma nova versão ou resolução, removendo os chunks do documento substituído. Para decisões com prazo, a resposta deve recomendar a confirmação com a coordenação ou o DRCA.